# Assignment 4: Semantic Search on Vision 2030 PDF

**Goal:** Build a semantic search pipeline:
1. **Load** the Vision 2030 PDF
2. **Split** it into chunks
3. **Embed** chunks using HuggingFace sentence-transformers
4. **Store** in an in-memory vector store
5. **Retrieve** relevant chunks for natural language queries

## 1. Load the PDF

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = os.path.join(os.getcwd(), "vision2030.pdf")
loader = PyPDFLoader(pdf_path)
docs = loader.load()

print(f"Loaded {len(docs)} pages from Vision 2030 PDF")
print(f"\n-- First page preview (300 chars) --")
print(docs[0].page_content[:300])
print(f"\n-- Metadata --")
print(docs[0].metadata)

c:\Users\Yaz00\Assignments_SDAIA_course\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\Yaz00\Assignments_SDAIA_course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Ignoring wrong pointing object 261 0 (offset 0)
Ignoring wrong pointing object 296 0 (offset 0)
Ignoring wrong pointing object 343 0 (offset 0)


Loaded 85 pages from Vision 2030 PDF

-- First page preview (300 chars) --


-- Metadata --
{'producer': 'Mac OS X 10.11.6 Quartz PDFContext', 'creator': 'Adobe InDesign CC 2015 (Macintosh)', 'creationdate': "D:20171121231749Z00'00'", 'moddate': "D:20171121231749Z00'00'", 'source': 'c:\\Users\\Yaz00\\Assignments_SDAIA_course\\Assignment_4\\vision2030.pdf', 'total_pages': 85, 'page': 0, 'page_label': '1'}


## 2. Split & Embed

In [2]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)
print(f"Split into {len(all_splits)} chunks")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

sample_vector = embeddings.embed_query(all_splits[0].page_content)
print(f"Embedding dimension: {len(sample_vector)}")
print(f"   First 5 values: {sample_vector[:5]}")

Split into 148 chunks


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2732.95it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 768
   First 5 values: [0.08075663447380066, 0.09103275090456009, -0.008611232042312622, 0.03532365337014198, 0.014396543614566326]


## 3. Store in Vector Store

In [3]:
vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(documents=all_splits)
print(f"Stored {len(ids)} chunks in the vector store")

Stored 148 chunks in the vector store


## 4. Semantic Search Results

In [4]:
queries = [
    "What is the main goal of Saudi Vision 2030?",
    "How will the economy be diversified away from oil?",
    "What are the plans for tourism in Saudi Arabia?",
    "How is education being reformed?",
    "What is the role of the private sector?",
]

print("=" * 70)
print("SEMANTIC SEARCH RESULTS")
print("=" * 70)

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 50)

    results = vector_store.similarity_search(query, k=2)
    for i, doc in enumerate(results, 1):
        print(f"  [{i}] (page: {doc.metadata.get('page', '?')})")
        print(f"      {doc.page_content[:150]}...")

    scored_results = vector_store.similarity_search_with_score(query, k=1)
    doc, score = scored_results[0]
    print(f"  Best match score: {score:.4f}")

SEMANTIC SEARCH RESULTS

Query: What is the main goal of Saudi Vision 2030?
--------------------------------------------------
  [1] (page: 5)
      flourishes, driving healthier employment opportunities 
for citizens and long-term prosperity for all. This 
promise is built on cooperation and on mu...
  [2] (page: 82)
      OUR COMMITMENT TO 
ACHIEVING THE GOALS 
OF THESE PIVOTAL 
PROGRAMS AND OUR 
COLLECTIVE 
CONTRIBUTION SHALL BE 
THE FIRST STEP 
TOWARDS ACHIEVING 
SAUD...
  Best match score: 0.8472

Query: How will the economy be diversified away from oil?
--------------------------------------------------
  [1] (page: 40)
      DIVERSIFYING OUR ECONOMY IS VITAL 
FOR ITS SUSTAINABILITY. ALTHOUGH OIL 
AND GAS ARE ESSENTIAL PILLARS OF 
OUR ECONOMY, WE HAVE BEGUN 
EXPANDING OUR I...
  [2] (page: 57)
      and new cross-border infrastructure projects, 
including land transport projects with Africa 
through Egypt. Logistical and trade exchanges will 
be s...
  Best match score: 0.5590

Q

## 5. Retriever Batch Query

In [5]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

batch_results = retriever.batch([
    "What are the healthcare goals?",
    "How will housing be improved?",
])

print("=" * 70)
print("RETRIEVER BATCH RESULTS")
print("=" * 70)

for i, result_docs in enumerate(batch_results):
    print(f"\n  Query {i+1} -> page: {result_docs[0].metadata.get('page', '?')}")
    print(f"    {result_docs[0].page_content[:150]}...")

RETRIEVER BATCH RESULTS

  Query 1 -> page: 27
    in the longer term. We will work towards developing 
private medical insurance to improve access to 
medical services and reduce waiting times for 
ap...

  Query 2 -> page: 55
    cities and improving its quality. Our specific goal is to 
exceed 90 percent housing coverage in densely 
populated cities and 66 percent in other urb...
